In [ ]:
%load_ext autoreload
%autoreload 2
%load_ext line_profiler

In [ ]:
from tqdm import tqdm

import os
import torch
import numpy as np
from torch_geometric.data import Batch, HeteroData
from numpy.linalg import LinAlgError

from utils.evaluation import solve_sdp_cvxpy
from torch_geometric.utils import to_dense_adj

In [ ]:
rng = np.random.RandomState(1)

In [ ]:
root = 'datasets/gen_edge_10'
os.mkdir(root)
os.mkdir(os.path.join(root, 'processed'))

In [ ]:
import networkx as nx
from torch_geometric.utils.convert import from_networkx


def erdos_renyi_generator(rng, n_min=100, n_max=100, p_min=0.15, p_max=0.15):
    n = rng.randint(n_min, n_max + 1)
    p = rng.uniform(p_min, p_max)
    G = nx.erdos_renyi_graph(n, p)
    return from_networkx(G)


def barabasi_albert_generator(rng, n_min=100, n_max=100, m_min=4, m_max=4):
    n = rng.randint(n_min, n_max + 1)
    m = rng.randint(n_min, n_max + 1)
    G = nx.barabasi_albert_graph(n, m)
    return from_networkx(G)

### synthetic WL

In [ ]:
def generate_gen_sdp(N, density):
    C = np.zeros((N, N))
    ar = np.arange(N)
    for _ in range(int(N * density) // 2):
        perm = rng.permutation(N)
        C[perm, ar] = 1
        C[ar, perm] = 1

    A = np.eye(N)[..., None]
    b = np.ones(1) * N * density
    return C.astype(np.float32), A.astype(np.float32), np.array(b, dtype=np.float32)

### synthetic Edge

In [ ]:
def generate_gen2_sdp(N):
    """
    e.g. N = 9, and creates 9 constraints
    solve with 1.e-1, 'mosek'
    """
    assert N % 2
    C = np.ones((N, N))
    # m = N if N % 2 else N - 1
    As = []

    row, col = np.triu_indices(N, 1)
    idx = np.random.permutation(row.shape[0])
    row = row[idx]
    col = col[idx]
    rows, cols = np.array_split(row, N), np.array_split(col, N)

    for i, (r, c) in enumerate(zip(rows, cols)):
        A = np.zeros((N, N))
        A[r, c] = 1
        A[c, r] = 1
        A[i, i] = 1
        if i == 0:
            # bias = np.random.randn()
            A[r[0], c[0]] += 1
            A[c[0], r[0]] += 1
        As.append(A)

    A = np.stack(As, axis=-1)
    b = np.ones(A.shape[-1])
    return C.astype(np.float32), A.astype(np.float32), np.array(b, dtype=np.float32)

### Max cut

In [ ]:
def generate_max_cut_sdp(nnodes, density):
    data = erdos_renyi_generator(rng, nnodes, nnodes, density, density)
    N = nnodes
    edge_index = data.edge_index
    E = edge_index.shape[1]
    adj = to_dense_adj(edge_index, max_num_nodes=N)[0].numpy()

    A = []
    b = []
    # diagonals being 1
    for i in range(N):
        const = np.zeros((N, N))
        const[i, i] = 1
        A.append(const)
        b.append(1)

    return adj, np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

### MIS

In [ ]:
def generate_mis_sdp(nnodes, density):
    data = erdos_renyi_generator(rng, nnodes, nnodes, density, density)
    N = nnodes
    edge_index = data.edge_index
    E = edge_index.shape[1]

    A = []
    b = []
    # edge = 0
    for i in range(E):
        x, y = edge_index[:, i].tolist()
        const = np.zeros((N, N))
        const[x, y] = 1
        A.append(const)
        b.append(0)
    # trace = 1
    A.append(np.eye(N))
    b.append(1)

    return -np.ones((N, N), dtype=np.float32), np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

### vertex cover

In [ ]:
def generate_cover_sdp(nnodes, density):
    data = erdos_renyi_generator(rng, nnodes, nnodes, density, density)
    N = nnodes
    edge_index = data.edge_index
    E = edge_index.shape[1]

    A = []
    b = []
    for i in range(E):
        x, y = edge_index[:, i].tolist()
        const = np.zeros((N + 1, N + 1))
        if x <= y:
            const[x, y] = 1
            const[0, x] = -1
            const[0, y] = -1
        else:
            const[x, y] = 1
            const[x, 0] = -1
            const[y, 0] = -1
        A.append(const)
        b.append(-1)

    for i in range(N+1):
        const = np.zeros((N+1, N+1))
        const[i, i] = 1
        A.append(const)
        b.append(1)

    C  = np.zeros((N+1, N+1), dtype=np.float32)
    C[0, :] = 1
    C[:, 0] = 1

    return C, np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

### Max 2 SAT

In [ ]:
def generate_2sat_sdp(klause, var):
    clause_vars = np.zeros((klause, 2), dtype=int)
    for i in range(klause):
        clause_vars[i] = np.random.choice(var, size=2, replace=False)
    
    signs = np.random.choice([-1, 1], size=(klause, 2), replace=True)
    
    # the clause matrix
    rows = np.repeat(np.arange(klause), 2)
    cols = clause_vars.flatten()
    data = signs.flatten()
    
    M = np.zeros((klause, var), dtype=int)
    M[rows, cols] = data
    MM = M.T @ M
    C = np.block([[MM - np.diag(np.diag(MM)), -M.sum(0)[:, None]], 
                  [-M.sum(0)[None], np.zeros((1, 1))]])

    C /= np.abs(C).max()

    A = []
    b = []
    # diagonals being 1
    for i in range(var + 1):
        const = np.zeros((var + 1, var + 1))
        const[i, i] = 1
        A.append(const)
        b.append(1)

    return C.astype(np.float32), np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

## max 3 SAT

In [ ]:
def generate_3sat_sdp(n_clauses, n_vars):
    dim = 2 * n_vars + 1  # v0 + x1..xn + ¬x1..¬xn

    # Random 3-literal clauses
    clause_vars = np.random.choice(n_vars, size=(n_clauses, 3), replace=True)
    signs = np.random.choice([1, -1], size=(n_clauses, 3))  # 1 = x, -1 = ¬x

    C = np.zeros((dim, dim))
    for idx in range(n_clauses):
        clause = clause_vars[idx]
        sign = signs[idx]

        # Map literals to vector indices
        def var_idx(var, polarity):
            return var + 1 if polarity == 1 else n_vars + var + 1

        a,b,c = [var_idx(clause[m], sign[m]) for m in range(3)]

        # Expand relaxations symbolically
        term = np.zeros((dim, dim))
        term[0, a] = 1
        term[0, b] = 1
        term[0, c] = 1
        term[a, b] = 1
        term[a, c] = 1
        term[b, c] = 1
        term += term.T  # Symmetric

        C -= term

    # Constraint matrices A and RHS vector b
    A = []
    b = []

    # Diagonal constraints: X_ii = 1 for all i
    for i in range(n_vars + 1):
        A_i = np.zeros((n_vars + 1, n_vars + 1))
        A_i[i, i] = 1
        A.append(A_i)
        b.append(1)

    C[1:1 + n_vars, 1:1 + n_vars] = C[1:1 + n_vars, 1:1 + n_vars] + \
                                    C[1 + n_vars:1 + 2*n_vars, 1 + n_vars:1 + 2*n_vars] - \
                                    C[1:1 + n_vars, 1 + n_vars:1 + 2*n_vars] - \
                                    C[1 + n_vars:1 + 2*n_vars, 1:1 + n_vars]
    C = C[:1 + n_vars, :1 + n_vars]
    C /= np.abs(C).max()

    return C.astype(np.float32), np.stack(A, axis=-1).astype(np.float32), np.array(b, dtype=np.float32)

# create ineq

In [ ]:
from cvxpy import DCPError, DGPError, DPPError, SolverError

In [ ]:
fnorm_strength = 1.e-5
solver = 'scs'

In [ ]:
graphs = []
pkg_idx = 0
success_cnt = 0

max_iter = 12000
num = 10000

pbar = tqdm(range(max_iter))
for i in pbar:
    C, A, b = generate_gen2_sdp(9)
    try:
        assert np.any(C != 0.)
        sol, X, stat, times = solve_sdp_cvxpy(C, A, b, fnorm_strength, solver)
        assert stat == 'optimal'
    except (LinAlgError, DCPError, DGPError, DPPError, SolverError, AssertionError):
        continue

    else:
        m = b.shape[0]
        n = C.shape[0]
        A = torch.from_numpy(A).float()
        A = A.reshape(-1, A.shape[-1]).T  # m, n**2
        A_where = torch.where(A)
        
        c2v_idx = torch.vstack(A_where)
        c2v_value = A[A_where][:, None]
        
        C = torch.from_numpy(C).float().reshape(-1)[None]
        # sparse vals obj connections
        C_where = torch.where(C)
        o2v_idx = torch.vstack(C_where)
        o2v_value = C[C_where][:, None]

        x = torch.from_numpy(X).float().reshape(-1)

        data = HeteroData(
            cons={
                'num_nodes': m,
                'x': torch.empty(m, 0),
                 },
            vals={
                'num_nodes': n ** 2,
                'x': torch.empty(n ** 2, 0),
            },
            obj={
                'num_nodes': 1,
                'x': torch.ones(1).float(),
                 },
            cons__to__vals={'edge_index': c2v_idx,
                            'edge_attr': c2v_value},
            obj__to__vals={'edge_index': o2v_idx,
                            'edge_attr': o2v_value},
            x_solution=x,
            obj_solution=torch.tensor([sol]),
            b=torch.from_numpy(b).float(),
        )
        success_cnt += 1
        graphs.append(data)

    if len(graphs) >= 1000 or success_cnt == num:
        torch.save(Batch.from_data_list(graphs), f'{root}/processed/batch{pkg_idx}.pt')
        pkg_idx += 1
        graphs = []

    if success_cnt >= num:
        break

    pbar.set_postfix({'suc': success_cnt})

## save as test only

In [ ]:
from torch_geometric.data import InMemoryDataset

datas = torch.load(f'{root}/processed/batch0.pt')
datas = Batch.to_data_list(datas)
torch.save(InMemoryDataset().collate(datas), f'{root}/processed/test.pt')
torch.save(None, f'{root}/processed/train.pt')
torch.save(None, f'{root}/processed/valid.pt')

## save as normal dataset

In [ ]:
from data.dataset import LPDataset

ds = LPDataset(root, 'valid')
assert not torch.isnan(ds.data.obj_solution).any()

In [ ]:
ds.data.x_solution